# AXI test 1: does a bigger model stop corrupting the graph?
Same 60 tasks as the 3B run (seed 3). Only the model changes: **Qwen2.5-7B-Instruct in 4-bit**. Two runs: no rule in the prompt, then the rule stated.

**Before you start:** Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save. Then run each cell below **one at a time, top to bottom**, and wait for the green tick before the next one.

Time: about 5 minutes to load, then about 40 minutes per run. If Colab disconnects, reconnect, run cells 1 to 4 again, then run the run-cell again: it continues where it stopped.

Prediction and reading rule are written in `docs/SCALE_TEST_7B.md` (before this run).

### Cell 1: check the GPU (it must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-7b.zip`

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the 4-bit tools (about 1 minute)

In [ ]:
!pip -q install bitsandbytes accelerate

### Cell 4: load the model (downloads about 15 GB once; about 5 minutes). It prints the GPU memory used when done

In [ ]:
import sys, json, time, torch
sys.path.insert(0, "/content/axi/python")
from colab.colab_llm_experiment import build_hf_llm
from colab.colab_scale_run import run_regime, comparison_table, verdict
from axi.experiments.agent_loop import make_tasks, print_summary
from axi.experiments.resumable import run_resumable

MODEL = "Qwen/Qwen2.5-7B-Instruct"
llm, model = build_hf_llm(MODEL, "cuda", load_4bit=True)
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1))

### Cell 5: quick check with 3 tasks (about 2 minutes). If this fails, stop and send me the red error text

In [ ]:
smoke = run_resumable(llm, make_tasks(3, 3), False, "/content/smoke.jsonl", meta=dict(model=MODEL, smoke=True))
print_summary(smoke["summary"])
r0 = smoke["rows"][0]
print("first task:", r0["kind"], "| M1 text:", repr(r0["M1_constrained"]["text"])[:160])

### Cell 6: run 1 of 2, NO rule in the prompt (about 40 minutes). The counter shows task i/60

In [ ]:
res_norule, path_norule = run_regime(llm, MODEL, "/content/out", hint_rules=False)
print()
print_summary(res_norule["summary"])
files.download(path_norule)

### Cell 7: run 2 of 2, rule STATED in the prompt (about 40 minutes)

In [ ]:
res_rule, path_rule = run_regime(llm, MODEL, "/content/out", hint_rules=True)
print()
print_summary(res_rule["summary"])
files.download(path_rule)

### Cell 8: the comparison with the 3B model, and the plain-words verdict

In [ ]:
results = {False: res_norule, True: res_rule}
print(comparison_table(results))
print()
print(verdict(results))

### Done
Two files were downloaded: `axi_Qwen2.5-7B-Instruct_seed3_norule.json` and `axi_Qwen2.5-7B-Instruct_seed3_rule.json`. Upload both to the chat. Do not paste any keys anywhere.